# 费城联储 ADS 实时版本原件预检

一次固定官方 ZIP GET，原始字节、SHA-256、NAS 远端哈希和限定恢复；只查看内部成员身份，不读 SOXX 结果或拟合模型。

In [ ]:
from pathlib import Path,PurePosixPath
from datetime import datetime,timezone
import hashlib,io,json,re,shlex,subprocess,tarfile,tempfile,zipfile
from src.data.nas_market_data import _settings,nas_api_call,_ssh
root=Path.cwd();card=root/'research/philadelphia_ads_vintage_source_20261023';plan_path=card/'plan.json';plan=json.loads(plan_path.read_text())
assert plan['frozen'] and not (card/'run_state.json').exists()
rawdir=root/'data/snapshots/philadelphia_ads_vintage_20261023';rawdir.mkdir(parents=True,exist_ok=True)
(card/'run_state.json').write_text(json.dumps({'status':'exact_official_ZIP_GET_started','plan_sha256':hashlib.sha256(plan_path.read_bytes()).hexdigest(),'GET_attempts':1,'model_fits':0,'accounts':0},indent=2)+'\n')
with tempfile.TemporaryDirectory(prefix='ads-vintages-') as tmp:
 headers=Path(tmp)/'headers.txt';download=Path(tmp)/'source.zip';started=datetime.now(timezone.utc).isoformat()
 command=['curl','--http1.1','--noproxy','*','--connect-timeout','10','--max-time','90','--fail-with-body','-sS','-L','-D',str(headers),'-o',str(download),'-H','Accept: application/zip','-A','SOXL-Qlib-source-audit/1.0',plan['fixed_ZIP_URL']]
 completed=subprocess.run(command,capture_output=True,text=True,timeout=100);received=datetime.now(timezone.utc).isoformat()
 receipt={'url':plan['fixed_ZIP_URL'],'requested_at_utc':started,'received_at_utc':received,'curl_exit_code':completed.returncode,'curl_stderr':completed.stderr[:1000],'response_headers':headers.read_text(errors='replace') if headers.exists() else '', 'bytes':download.stat().st_size if download.exists() else 0}
 (card/'capture_receipt.json').write_text(json.dumps(receipt,indent=2)+'\n')
 if completed.returncode:raise RuntimeError('single frozen ADS ZIP GET failed; no retry')
 raw=download.read_bytes();assert len(raw)<=plan['budget']['max_ZIP_bytes'] and zipfile.is_zipfile(io.BytesIO(raw))
 source=rawdir/'ADS_All_Vintages.zip';source.write_bytes(raw);source_sha=hashlib.sha256(raw).hexdigest()
 with zipfile.ZipFile(io.BytesIO(raw)) as z:
  members=[]
  for item in z.infolist():
   name=PurePosixPath(item.filename)
   assert not name.is_absolute() and '..' not in name.parts and not item.is_dir()
   members.append({'name':item.filename,'uncompressed_bytes':item.file_size,'compressed_bytes':item.compress_size,'crc32':item.CRC})
  assert members
 settings=_settings(root);archive=Path(tmp)/'original_source.tar.gz'
 with tarfile.open(archive,'w:gz') as out:out.add(source,arcname='ADS_All_Vintages.zip')
 digest=hashlib.sha256(archive.read_bytes()).hexdigest();size=archive.stat().st_size
 evidence=nas_api_call(origin=settings['origin'],token=settings['token'],method='PUT',path='/api/research/artifacts/'+digest,upload=archive,executor_id='soxl-minute-research')['artifact']
 assert evidence['sha256']==digest and evidence['size_bytes']==size
 remote=settings['artifact_root'].rstrip('/')+'/'+digest[:2]+'/'+digest[2:4]+'/'+digest
 assert _ssh(settings['ssh'],'sha256sum '+shlex.quote(remote)).split()[0]==digest
 cmd=['ssh','-p',str(settings['ssh']['port']),'-o','BatchMode=yes','-o','ConnectTimeout=5',settings['ssh']['host'],'tar -xOzf '+repr(remote)+' ADS_All_Vintages.zip']
 restored=subprocess.run(cmd,capture_output=True,timeout=120);assert restored.returncode==0 and hashlib.sha256(restored.stdout).hexdigest()==source_sha and len(restored.stdout)==len(raw)
 result={'status':'official_ADS_all_vintages_ZIP_bytes_archived_schema_preflight','plan_sha256':hashlib.sha256(plan_path.read_bytes()).hexdigest(),'raw_ZIP_sha256':source_sha,'raw_ZIP_bytes':len(raw),'members':members,'nas_archive_sha256':digest,'nas_archive_bytes':size,'nas_artifact':evidence,'nas_remote_hash_verified':True,'scoped_raw_ZIP_restore_verified':True,'vintage_dates_qualified':False,'market_outcome_reads':0,'Qlib_factor_rows':0,'model_fits':0,'native_accounts':0,'SOXX_goal_achieved':False}
 (card/'source_preflight_result.json').write_text(json.dumps(result,indent=2)+'\n')
 (card/'run_state.json').write_text(json.dumps({'status':'completed_source_bytes_schema_pending','plan_sha256':result['plan_sha256'],'GETs':1,'model_fits':0,'accounts':0},indent=2)+'\n')
 print({'status':result['status'],'raw_ZIP_bytes':len(raw),'members':[x['name'] for x in members]})

## 结论边界

ZIP 字节取得只证明来源可访问。版本字段、发布日期、2011–2026 覆盖、改订和 Stage1 增量需要后续限定审查。